<a href="https://colab.research.google.com/github/regenpalkar28/Smoke-Simulation-PC-Project/blob/main/PC_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi
!nvcc --version

In [ ]:
%%writefile fluid_solver.cu

#include <iostream>
#include <vector>
#include <chrono>
#include <cmath>
#include <fstream>
#include <cuda_runtime.h>

#define N 256
#define SIZE ((N + 2) * (N + 2))
#define IX(i, j) ((i) + (N + 2) * (j))
#define SWAP(x0, x) { float *tmp = x0; x0 = x; x = tmp; }

#define CUDA_CHECK(call) \
    do { \
        cudaError_t err = call; \
        if (err != cudaSuccess) { \
            std::cerr << "CUDA Error: " << cudaGetErrorString(err) << " at line " << __LINE__ << std::endl; \
            exit(EXIT_FAILURE); \
        } \
    } while (0)

__global__ void set_bnd_kernel(int n, int b, float *x) {
    int i = blockIdx.x * blockDim.x + threadIdx.x + 1;
    if (i <= n) {
        x[IX(0, i)]     = b == 1 ? -x[IX(1, i)] : x[IX(1, i)];
        x[IX(N + 1, i)] = b == 1 ? -x[IX(N, i)] : x[IX(N, i)];
        x[IX(i, 0)]     = b == 2 ? -x[IX(i, 1)] : x[IX(i, 1)];
        x[IX(i, N + 1)] = b == 2 ? -x[IX(i, N)] : x[IX(i, N)];
    }
}

void set_bnd_gpu(int n, int b, float *d_x) {
    int threads = 256;
    int blocks = (n + threads - 1) / threads;
    set_bnd_kernel<<<blocks, threads>>>(n, b, d_x);
}

// 1. Continuous High-Volume Smoke & Velocity Emitter
__global__ void inject_smoke_kernel(int n, float *d_dens, float *d_v, float *d_u, float dt, int step) {
    int i = blockIdx.x * blockDim.x + threadIdx.x + 1;
    int j = blockIdx.y * blockDim.y + threadIdx.y + 1;

    if (i <= n && j <= n) {
        // Wide smoke jet base across the bottom
        if (i >= (n/2 - 35) && i <= (n/2 + 35) && j >= 2 && j <= 18) {
            d_dens[IX(i, j)] += 1200.0f * dt;
            d_v[IX(i, j)]    += 800.0f * dt;

            // Dynamic oscillating horizontal turbulence
            float perturb = sinf(i * 0.2f + step * 0.15f) * 150.0f;
            d_u[IX(i, j)] += perturb * dt;
        }

        // Global Thermal Buoyancy Force
        float density = d_dens[IX(i, j)];
        if (density > 0.1f) {
            d_v[IX(i, j)] += density * 0.35f * dt;
        }
    }
}

// 2. Vorticity Confinement Kernel (Restores lost rotational energy & micro-swirls)
__global__ void vorticity_confinement_kernel(int n, float *u, float *v, float dt) {
    int i = blockIdx.x * blockDim.x + threadIdx.x + 1;
    int j = blockIdx.y * blockDim.y + threadIdx.y + 1;

    if (i >= 2 && i <= n - 1 && j >= 2 && j <= n - 1) {
        // Compute local curl (vorticity magnitude)
        float dw_dx = (v[IX(i + 1, j)] - v[IX(i - 1, j)]) * 0.5f;
        float dw_dy = (u[IX(i, j + 1)] - u[IX(i, j - 1)]) * 0.5f;
        float curl = dw_dx - dw_dy;

        // Compute gradient of curl magnitude
        float curl_left  = fabsf((v[IX(i, j)] - v[IX(i - 2, j)]) * 0.5f - (u[IX(i - 1, j + 1)] - u[IX(i - 1, j - 1)]) * 0.5f);
        float curl_right = fabsf((v[IX(i + 2, j)] - v[IX(i, j)]) * 0.5f - (u[IX(i + 1, j + 1)] - u[IX(i + 1, j - 1)]) * 0.5f);
        float curl_down  = fabsf((v[IX(i + 1, j - 1)] - v[IX(i - 1, j - 1)]) * 0.5f - (u[IX(i, j)] - u[IX(i, j - 2)]) * 0.5f);
        float curl_up    = fabsf((v[IX(i + 1, j + 1)] - v[IX(i - 1, j + 1)]) * 0.5f - (u[IX(i, j + 2)] - u[IX(i, j)]) * 0.5f);

        float Nx = (curl_right - curl_left) * 0.5f;
        float Ny = (curl_up - curl_down) * 0.5f;
        float len = sqrtf(Nx * Nx + Ny * Ny) + 1e-5f;

        Nx /= len;
        Ny /= len;

        // Vorticity confinement force scale
        float epsilon = 2.5f;
        u[IX(i, j)] += Nx * curl * epsilon * dt;
        v[IX(i, j)] += Ny * curl * epsilon * dt;
    }
}

__global__ void advect_kernel(int n, int b, float *d, const float *d0, const float *u, const float *v, float dt) {
    int i = blockIdx.x * blockDim.x + threadIdx.x + 1;
    int j = blockIdx.y * blockDim.y + threadIdx.y + 1;

    if (i <= n && j <= n) {
        float dt0 = dt * n;
        float x = (float)i - dt0 * u[IX(i, j)];
        float y = (float)j - dt0 * v[IX(i, j)];

        if (x < 0.5f) x = 0.5f;
        if (x > n + 0.5f) x = n + 0.5f;
        int i0 = (int)x;
        int i1 = i0 + 1;

        if (y < 0.5f) y = 0.5f;
        if (y > n + 0.5f) y = n + 0.5f;
        int j0 = (int)y;
        int j1 = j0 + 1;

        float s1 = x - i0;
        float s0 = 1.0f - s1;
        float t1 = y - j0;
        float t0 = 1.0f - t1;

        d[IX(i, j)] = s0 * (t0 * d0[IX(i0, j0)] + t1 * d0[IX(i0, j1)]) +
                      s1 * (t0 * d0[IX(i1, j0)] + t1 * d0[IX(i1, j1)]);
    }
}

__global__ void divergence_kernel(int n, const float *u, const float *v, float *div, float *p) {
    int i = blockIdx.x * blockDim.x + threadIdx.x + 1;
    int j = blockIdx.y * blockDim.y + threadIdx.y + 1;

    if (i <= n && j <= n) {
        div[IX(i, j)] = -0.5f * (u[IX(i + 1, j)] - u[IX(i - 1, j)] +
                                 v[IX(i, j + 1)] - v[IX(i, j - 1)]) / n;
        p[IX(i, j)] = 0.0f;
    }
}

__global__ void jacobi_kernel(int n, float *p_out, const float *p_in, const float *div) {
    int i = blockIdx.x * blockDim.x + threadIdx.x + 1;
    int j = blockIdx.y * blockDim.y + threadIdx.y + 1;

    if (i <= n && j <= n) {
        p_out[IX(i, j)] = (div[IX(i, j)] + p_in[IX(i - 1, j)] + p_in[IX(i + 1, j)] +
                           p_in[IX(i, j - 1)] + p_in[IX(i, j + 1)]) / 4.0f;
    }
}

__global__ void project_grad_kernel(int n, float *u, float *v, const float *p) {
    int i = blockIdx.x * blockDim.x + threadIdx.x + 1;
    int j = blockIdx.y * blockDim.y + threadIdx.y + 1;

    if (i <= n && j <= n) {
        u[IX(i, j)] -= 0.5f * n * (p[IX(i + 1, j)] - p[IX(i - 1, j)]);
        v[IX(i, j)] -= 0.5f * n * (p[IX(i, j + 1)] - p[IX(i, j - 1)]);
    }
}

void project_gpu(int n, float *d_u, float *d_v, float *d_p, float *d_div, float *d_p_next) {
    dim3 threadsPerBlock(16, 16);
    dim3 numBlocks((n + 15) / 16, (n + 15) / 16);

    divergence_kernel<<<numBlocks, threadsPerBlock>>>(n, d_u, d_v, d_div, d_p);
    set_bnd_gpu(n, 0, d_div);
    set_bnd_gpu(n, 0, d_p);

    for (int k = 0; k < 40; k++) {
        jacobi_kernel<<<numBlocks, threadsPerBlock>>>(n, d_p_next, d_p, d_div);
        set_bnd_gpu(n, 0, d_p_next);
        SWAP(d_p, d_p_next);
    }

    project_grad_kernel<<<numBlocks, threadsPerBlock>>>(n, d_u, d_v, d_p);
    set_bnd_gpu(n, 1, d_u);
    set_bnd_gpu(n, 2, d_v);
}

void advect_gpu(int n, int b, float *d_d, const float *d_d0, const float *d_u, const float *d_v, float dt) {
    dim3 threadsPerBlock(16, 16);
    dim3 numBlocks((n + 15) / 16, (n + 15) / 16);

    advect_kernel<<<numBlocks, threadsPerBlock>>>(n, b, d_d, d_d0, d_u, d_v, dt);
    set_bnd_gpu(n, b, d_d);
}

int main() {
    size_t mem_size = SIZE * sizeof(float);
    std::vector<float> h_dens(SIZE, 0.0f);

    float *d_u, *d_v, *d_u0, *d_v0, *d_dens, *d_dens0, *d_p, *d_div, *d_p_next;
    CUDA_CHECK(cudaMalloc((void**)&d_u, mem_size));
    CUDA_CHECK(cudaMalloc((void**)&d_v, mem_size));
    CUDA_CHECK(cudaMalloc((void**)&d_u0, mem_size));
    CUDA_CHECK(cudaMalloc((void**)&d_v0, mem_size));
    CUDA_CHECK(cudaMalloc((void**)&d_dens, mem_size));
    CUDA_CHECK(cudaMalloc((void**)&d_dens0, mem_size));
    CUDA_CHECK(cudaMalloc((void**)&d_p, mem_size));
    CUDA_CHECK(cudaMalloc((void**)&d_div, mem_size));
    CUDA_CHECK(cudaMalloc((void**)&d_p_next, mem_size));

    CUDA_CHECK(cudaMemset(d_u, 0, mem_size));
    CUDA_CHECK(cudaMemset(d_v, 0, mem_size));
    CUDA_CHECK(cudaMemset(d_u0, 0, mem_size));
    CUDA_CHECK(cudaMemset(d_v0, 0, mem_size));
    CUDA_CHECK(cudaMemset(d_dens, 0, mem_size));
    CUDA_CHECK(cudaMemset(d_dens0, 0, mem_size));

    float dt = 0.12f;
    int num_steps = 300;

    dim3 threadsPerBlock(16, 16);
    dim3 numBlocks((N + 15) / 16, (N + 15) / 16);

    std::ofstream raw_file("density_frames.bin", std::ios::binary);

    std::cout << "Executing CUDA High-Fidelity Smoke Acceleration Solver..." << std::endl;
    auto start_time = std::chrono::high_resolution_clock::now();

    for (int step = 0; step < num_steps; step++) {
        inject_smoke_kernel<<<numBlocks, threadsPerBlock>>>(N, d_dens, d_v, d_u, dt, step);
        vorticity_confinement_kernel<<<numBlocks, threadsPerBlock>>>(N, d_u, d_v, dt);

        SWAP(d_u0, d_u);
        SWAP(d_v0, d_v);
        advect_gpu(N, 1, d_u, d_u0, d_u0, d_v0, dt);
        advect_gpu(N, 2, d_v, d_v0, d_u0, d_v0, dt);
        project_gpu(N, d_u, d_v, d_p, d_div, d_p_next);

        SWAP(d_dens0, d_dens);
        advect_gpu(N, 0, d_dens, d_dens0, d_u, d_v, dt);

        CUDA_CHECK(cudaMemcpy(h_dens.data(), d_dens, mem_size, cudaMemcpyDeviceToHost));
        raw_file.write(reinterpret_cast<char*>(h_dens.data()), mem_size);
    }

    raw_file.close();
    auto end_time = std::chrono::high_resolution_clock::now();
    double total_ms = std::chrono::duration<double, std::milli>(end_time - start_time).count();

    std::cout << "Simulation Done! Time: " << total_ms << " ms | FPS: " << (num_steps / (total_ms / 1000.0)) << std::endl;

    cudaFree(d_u); cudaFree(d_v); cudaFree(d_u0); cudaFree(d_v0);
    cudaFree(d_dens); cudaFree(d_dens0); cudaFree(d_p); cudaFree(d_div); cudaFree(d_p_next);

    return 0;
}

In [ ]:
!nvcc -O3 fluid_solver.cu -o fluid_solver
!./fluid_solver

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import HTML, display

# Set the animation embedding limit higher (e.g., 50 MB) to prevent the warning
plt.rcParams['animation.embed_limit'] = 50.0

N = 256
size = (N + 2) * (N + 2)

data = np.fromfile("density_frames.bin", dtype=np.float32)
num_frames = len(data) // size
frames = data.reshape((num_frames, N + 2, N + 2))

# Extract non-boundary inner grid domain
frames = frames[:, 1:N+1, 1:N+1]

# Dynamic Crop focused on full smoke plume development
cropped_frames = frames[:, 0:240, 20:236]

# Custom Realistic Volumetric Smoke Color Map (Deep space dark -> Charcoal -> Light Gray -> White core)
smoke_colors = [
    (0.00, 0.00, 0.00), # Vacuum background
    (0.08, 0.08, 0.10), # Dark outer tendril edges
    (0.35, 0.35, 0.38), # Mid-density smoke body
    (0.70, 0.70, 0.72), # Dense smoke plume
    (0.95, 0.95, 0.98), # Bright hot source core
]
smoke_cmap = LinearSegmentedColormap.from_list("realistic_smoke", smoke_colors)

# Compute optimal dynamic contrast scale
vmax_cap = np.percentile(cropped_frames[-1], 97)
if vmax_cap <= 0:
    vmax_cap = 1.0

fig, ax = plt.subplots(figsize=(8, 10), facecolor='#050508')
ax.set_facecolor('#050508')

im = ax.imshow(
    cropped_frames[0],
    cmap=smoke_cmap,
    origin="lower",
    animated=True,
    vmin=0.0,
    vmax=vmax_cap
)

ax.set_title("CUDA Real-Time Eulerian Smoke Dynamics (Vorticity Accelerated)", color="white", fontsize=11, pad=12)
ax.axis("off")
fig.tight_layout()

def update(frame_idx):
    im.set_array(cropped_frames[frame_idx])
    return [im]

ani = animation.FuncAnimation(fig, update, frames=num_frames, interval=25, blit=True)
plt.close()

# Export clean inline MP4 video render
display(HTML(ani.to_html5_video()))